In [ ]:
# Cell 1: Imports and setup
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 2: Extract and load your downloaded CALTECH-101 dataset
from zipfile import ZipFile

ZIP_PATH = Path(r'C:\Users\User\Desktop\clases\edgeai\Midterm\archive.zip')
DATA_DIR = Path.cwd() / 'data' / 'caltech-101'
if not DATA_DIR.is_dir():
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f'Dataset ZIP not found: {ZIP_PATH}')
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
print(f'Dataset folder: {DATA_DIR}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
images = datasets.ImageFolder(DATA_DIR, transform=transform)  # Converts images to RGB.
# CALTECH-101 includes a background folder that is not an object category.
class_names = [name for name in images.classes if name.lower() != 'background_google']
label_map = {images.class_to_idx[name]: i for i, name in enumerate(class_names)}
images.target_transform = lambda target: label_map[target]
dataset = Subset(images, [i for i, (_, target) in enumerate(images.samples) if target in label_map])
num_classes = len(class_names)
if num_classes < 2 or len(dataset) < 5:
    raise ValueError('The dataset must contain at least two categories and five images.')
print(f'Loaded {len(dataset):,} images across {num_classes} categories.')

In [ ]:
# Cell 3: Residual block: Conv -> BatchNorm -> ReLU, then Conv -> BatchNorm + skip -> ReLU
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)
# Ordinary blocks use stride=1; stage transitions use stride=2 for downsampling.

In [ ]:
# Cell 4: ResNet-18 (eight residual blocks: [2, 2, 2, 2])
class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.layers = nn.Sequential(
            ResidualBlock(64, 64), ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128),
            ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256),
            ResidualBlock(256, 512, stride=2), ResidualBlock(512, 512),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(torch.flatten(self.pool(x), 1))

model = ResNet18(num_classes).to(device)
print(f'ResNet-18: {sum(p.numel() for p in model.parameters()):,} parameters')

In [ ]:
# Cell 5: Loss and optimizer
# ReLU is an activation, not a classification loss; it is used inside the network.
# Cross-entropy accepts raw class logits and integer category labels.
EPOCHS, LEARNING_RATE, BATCH_SIZE = 30, 1e-3, 64
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# Cell 6: Reproducible 80:20 training/validation split and 30-epoch training
train_size = int(0.8 * len(dataset))
train_set, val_set = random_split(
    dataset, [train_size, len(dataset) - train_size],
    generator=torch.Generator().manual_seed(SEED),
)
# num_workers=0 works with Windows and VS Code notebooks.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Training: {len(train_set):,}; validation: {len(val_set):,}')
history = {key: [] for key in ['train_loss', 'val_loss', 'train_acc', 'val_acc']}
best_acc, best_state = -1.0, None

for epoch in range(EPOCHS):
    for phase, loader in [('train', train_loader), ('val', val_loader)]:
        training = phase == 'train'
        model.train(training)
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                logits = model(inputs)
                loss = criterion(logits, labels)
                if training:
                    loss.backward()
                    optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        history[f'{phase}_loss'].append(total_loss / total)
        history[f'{phase}_acc'].append(correct / total)
    if history['val_acc'][-1] > best_acc:
        best_acc = history['val_acc'][-1]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | "
          f"loss {history['train_loss'][-1]:.4f}/{history['val_loss'][-1]:.4f} | "
          f"accuracy {history['train_acc'][-1]:.2%}/{history['val_acc'][-1]:.2%} (train/val)")

model.load_state_dict(best_state)
torch.save({'model_state_dict': best_state, 'class_names': class_names,
            'image_size': 224, 'best_val_accuracy': best_acc}, 'resnet18_caltech101_best.pt')
print(f'Best validation accuracy: {best_acc:.2%}; checkpoint saved.')
# The validation set selects the checkpoint; these are not independent test results.

In [ ]:
# Cell 7: Training and validation loss / accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)
for phase, label in [('train', 'Training'), ('val', 'Validation')]:
    axes[0].plot(epochs, history[f'{phase}_loss'], label=label)
    axes[1].plot(epochs, history[f'{phase}_acc'], label=label)
for ax, title in zip(axes, ['Cross-entropy loss', 'Accuracy']):
    ax.set(xlabel='Epoch', ylabel=title, title=title)
    ax.legend()
    ax.grid(alpha=0.3)
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Confusion matrix on validation data using the best checkpoint
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        predictions = model(inputs.to(device)).argmax(1).cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(predictions)
y_true, y_pred = np.array(y_true), np.array(y_pred)
cm = np.zeros((num_classes, num_classes), dtype=np.int64)
np.add.at(cm, (y_true, y_pred), 1)

fig, ax = plt.subplots(figsize=(22, 20))
plot = ax.imshow(cm, cmap='Blues', interpolation='nearest')
fig.colorbar(plot, ax=ax, label='Image count')
ax.set(xticks=np.arange(num_classes), yticks=np.arange(num_classes),
       xticklabels=class_names, yticklabels=class_names,
       xlabel='Predicted category', ylabel='True category',
       title='Validation confusion matrix (best checkpoint)')
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=6)
plt.setp(ax.get_yticklabels(), fontsize=6)
plt.tight_layout()
plt.savefig('resnet18_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9: Validation metrics table (macro averages across all categories)
tp = np.diag(cm).astype(float)
predicted_count, true_count = cm.sum(axis=0), cm.sum(axis=1)
precision = np.divide(tp, predicted_count, out=np.zeros_like(tp), where=predicted_count != 0)
recall = np.divide(tp, true_count, out=np.zeros_like(tp), where=true_count != 0)
f1 = np.divide(2 * precision * recall, precision + recall,
               out=np.zeros_like(tp), where=(precision + recall) != 0)
# MSE/MAE on arbitrary category IDs depend on label order; interpret with care.
# Here they are explicitly computed on integer class IDs to meet the requirement.
errors = y_pred.astype(float) - y_true.astype(float)
metrics = {
    'Accuracy': float(tp.sum() / cm.sum()),
    'Precision (macro)': float(precision.mean()),
    'Recall (macro)': float(recall.mean()),
    'F1-Score (macro)': float(f1.mean()),
    'MSE (class IDs)': float(np.mean(errors ** 2)),
    'MAE (class IDs)': float(np.mean(np.abs(errors))),
}
print(f"{'Metric':<24} {'Value':>12}")
print('-' * 37)
for name, value in metrics.items():
    print(f'{name:<24} {value:>12.6f}')
print('Undefined per-class ratios are set to 0; accuracy and macro scores range from 0 to 1.')